# Chapter 14 &mdash; A Recursive Language: $L_{EmptyDFA}$

**Concept 5 of the Chapter 14 decomposition:** *A Recursive Language: $L_{EmptyDFA}$, and the Definition of a Decider*

Emptiness of a DFA is algorithmically checkable, so the language of such descriptions is decidable.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-L-EmptyDFA-Is-Recursive/Concept-L-EmptyDFA-Is-Recursive.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


$$L_{EmptyDFA} = \{\langle D\rangle : D \text{ is a DFA and } L(D)=\emptyset\}$$

This is a language whose **strings are machine descriptions**. That shift &mdash; from
languages of data to languages of *programs* &mdash; is what the rest of the chapter is
about.

It is **recursive**, and the decider is a reachability search: mark $q_0$, close under
$\delta$, and answer "empty" iff no final state was marked. At most $|Q|$ rounds, so
it **always halts** &mdash; which is exactly the definition of a decider.

Note what makes it easy: a DFA is a *finite* object and the property is about its
*structure*. Concept 8 shows what happens when the property is about **behaviour**
instead.

## 2. Definitions

### The decider

In [ ]:
def decide_empty(D):
    seen, frontier, rounds = {D["q0"]}, {D["q0"]}, 0
    while frontier:
        rounds += 1
        frontier = {step_dfa(D, q, a) for q in frontier for a in D["Sigma"]} - seen
        seen |= frontier
    return (not (seen & D["F"])), rounds, seen

### Some DFA to test it on

In [ ]:
CASES = {
 'no final state'    : 'DFA\nI : 0 | 1 -> I\n',
 'final unreachable' : '''DFA
I  : 0 | 1 -> I
F1 : 0 | 1 -> F1
''',
 'final reachable'   : '''DFA
I : 0 -> F
I : 1 -> I
F : 0 | 1 -> F
''',
 'start is final'    : 'DFA\nIF : 0 | 1 -> IF\n',
}

## 3. Tests

The decider on each case.

In [ ]:
for name, src in CASES.items():
    D = md2mc(src)
    empty, rounds, seen = decide_empty(D)
    print("  %-18s empty? %-6s rounds %d, reachable %s"
          % (name, empty, rounds, sorted(seen)))
assert decide_empty(md2mc(CASES['no final state']))[0]
assert decide_empty(md2mc(CASES['final unreachable']))[0]
assert not decide_empty(md2mc(CASES['final reachable']))[0]

**A final state is not enough** &mdash; it must be reachable.

In [ ]:
D = md2mc(CASES['final unreachable'])
print("F =", sorted(D["F"]), " but reachable =", sorted(decide_empty(D)[2]))
assert D["F"] and decide_empty(D)[0]
print("\nThe machine has an accepting state it can never get to.")

It **always halts**: at most $|Q|$ rounds, whatever the machine.

In [ ]:
import random
worst = 0
for trial in range(30):
    n = random.randint(1, 7)
    names = ['I'] + ['S%d' % i for i in range(1, n)]
    lines = ['DFA'] + ['%s : %s -> %s' % (q, a, random.choice(names))
                       for q in names for a in '01']
    D = md2mc('\n'.join(lines))
    _, rounds, _ = decide_empty(D)
    worst = max(worst, rounds)
    assert rounds <= len(D["Q"]) + 1
print("30 random DFA decided; worst case %d rounds" % worst)
print("no fuel, no timeout, no 'not yet' -- that is what DECIDER means")

Cross-check against brute-force enumeration.

In [ ]:
from itertools import product
for name, src in CASES.items():
    D = md2mc(src)
    brute = not any(accepts_dfa(D, ''.join(p))
                    for k in range(9) for p in product('01', repeat=k))
    assert decide_empty(D)[0] == brute, name
print("decider agrees with brute force on every case")

Why this one is easy, and what will be hard.

In [ ]:
print("easy here : a DFA is FINITE, and emptiness is a question about its GRAPH")
print("hard later: the same question about a TM is about its BEHAVIOUR,")
print("            and behaviour is unbounded")

## 4. Animation

A DFA whose language is non-empty &mdash; the final state is reachable.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(md2mc(CASES['final reachable']), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Write a decider for $L_{UnivDFA} = \{\langle D\rangle : L(D)=\Sigma^*\}$.
2. Is DFA **equivalence** decidable? (Chapter 6 says yes &mdash; how?)
3. What breaks if you try the same reachability argument on a TM?

In [ ]:
# Your work for the exercises above.